In [221]:
import pandas as pd
import numpy as np

In [222]:
bb_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\bbb\combined_ipl_bbb.csv")
xi_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\register_data\ipl_playing_xi.csv")

In [223]:
# Ensure player_dismissed is numeric or handle potential errors
bb_df['player_dismissed'] = pd.to_numeric(bb_df['player_dismissed'], errors='coerce')

# Convert player IDs in playing XI from string representation to list of integers
for col in ['team1_playing_xi', 'team2_playing_xi']:
    # Convert string representation of lists to actual lists of integers
    xi_df[col] = xi_df[col].apply(lambda x: [int(p.strip('"')) for p in x.strip('[]').replace(' ', '').split(',') if p])

correcting some inconsistency in naming

In [224]:
print("Identifying player name inconsistencies...")
# Define the corrections: {batter_id: 'Correct Name'}
name_corrections = {
    4864: 'Dan Christian',          # Choosing the more common name
    27280: 'Manvinder Bisla',        # Removing extra space
    302208: 'CM Gautam',             # Choosing the shorter/common name
    322535: 'Karanveer Singh',       # Choosing the likely correct spelling
    326016: 'Bhuvneshwar Kumar',     # Correct spelling
    784379: 'Wanindu Hasaranga'      # Shorter/common name
}
print(f"Applying corrections for {len(name_corrections)} player IDs...")
corrections_applied_count = 0
# Loop through the identified player IDs and apply the standard name
for player_id, correct_name in name_corrections.items():
    # Create a mask for rows matching the current player ID
    mask = bb_df['batter_id'] == player_id

    # Check if any rows match before attempting update
    if mask.any():
        # Update the 'batter' column for these rows to the correct name
        bb_df.loc[mask, 'batter'] = correct_name
        corrections_applied_count += 1
        # Optional: You could print which IDs were updated
        # print(f"  - Updated batter name for ID {player_id} to '{correct_name}'")
    else:
        print(f"  - Warning: No rows found in bb_df for batter_id {player_id}. No correction applied.")

print(f"Name corrections applied for {corrections_applied_count} IDs found in the DataFrame.")

Identifying player name inconsistencies...
Applying corrections for 6 player IDs...
Name corrections applied for 6 IDs found in the DataFrame.


In [225]:
bb_df['dismissal'] = bb_df.apply(lambda row: 1 if pd.notnull(row['player_dismissed']) and row['player_dismissed'] == row['batter_id'] else 0, axis=1)

### TASK 1 - Batter Performance Metrics

In [226]:
# --- Step 0: Create helper mappings using bb_df ---

# Mapping for batter names: use the first non-null occurrence per batter_id.
batter_name_map = bb_df.dropna(subset=['batter']).groupby('batter_id')['batter'].first().to_dict()

# Mapping for match date: assume each match_id in bb_df has a consistent date, so we take the first date seen.
match_date_map = bb_df.groupby('match_id')['date'].first().to_dict()

# --- Step 1: Prepare the ball-by-ball data ---
# Create a dismissal indicator column: if the 'player_dismissed' equals 'batter_id', mark as 1, else 0.
bb_df['dismissal'] = bb_df.apply(
    lambda row: 1 if pd.notnull(row['player_dismissed']) and (row['player_dismissed'] == row['batter_id']) else 0,
    axis=1
)

# Define helper functions to identify boundaries.
def is_four(run):
    return 1 if run == 4 else 0

def is_six(run):
    return 1 if run == 6 else 0

# Aggregate ball-by-ball data for each match, batter and bowler type.
agg_bb = bb_df.groupby(['match_id', 'batter_id', 'bowl_style']).agg(
    total_runs=('batruns', 'sum'),
    total_balls=('ballfaced', 'sum'),
    fours=('batruns', lambda x: x.apply(is_four).sum()),
    sixes=('batruns', lambda x: x.apply(is_six).sum()),
    dismissals=('dismissal', lambda x: 1 if x.sum() >= 1 else 0) # Count dismissal as 1 for the match if any record shows a dismissal for that batter.
).reset_index()

# --- Step 2: Create a DataFrame of all match/player-bowl_style combinations ---
# Here, we use xi_df which should contain a column 'match_id', and columns for both teams' playing XI:
# 'team1_playing_xi' and 'team2_playing_xi'.
rows = []
for i, row in xi_df.iterrows():
    match_id = row['match_id']
    # If you have a date in xi_df, you could use it to overwrite the bb_df date mapping.
    # Here we rely on match_date_map obtained from bb_df.
    players = row['team1_playing_xi'] + row['team2_playing_xi']
    for batter in players:
        batter_int = int(batter)  # convert to int to ensure matching with bb_df's batter_id
        for style in ['RAF', 'LAF', 'RLB', 'ROB', 'SLA', 'LWS']:
            rows.append({'match_id': match_id, 'batter_id': batter_int, 'bowl_style': style})

all_combinations = pd.DataFrame(rows)

# Merge the aggregated ball-by-ball stats with all combinations.
match_stats = pd.merge(all_combinations, agg_bb, on=['match_id', 'batter_id', 'bowl_style'], how='left')
# Fill missing stats (when a batter did not bat against a bowling style) with 0.
match_stats[['total_runs', 'total_balls', 'fours', 'sixes', 'dismissals']] = match_stats[['total_runs', 'total_balls', 'fours', 'sixes', 'dismissals']].fillna(0)

# Add the date for each match from the mapping.
match_stats['date'] = match_stats['match_id'].map(match_date_map)

# --- Step 3: Compute cumulative performance metrics ---
# Sort match_stats by batter, bowling style and match (here using match_id order).
match_stats = match_stats.sort_values(['batter_id', 'bowl_style', 'match_id'])

def compute_cumulative_metrics(grp):
    rpb_list = []
    bdr_pct_list = []
    rpd_list = []
    bpd_list = []

    cum_runs = 0
    cum_balls = 0
    cum_fours = 0
    cum_sixes = 0
    cum_dismissals = 0

    for idx, row in grp.iterrows():
        # Compute metrics from all previous matches (if any).
        if cum_balls > 0:
            rpb = cum_runs / cum_balls
            bdr_pct = ((cum_fours + cum_sixes) / cum_balls) * 100
        else:
            rpb = 0
            bdr_pct = 0

        if cum_dismissals > 0:
            rpd = cum_runs / cum_dismissals
            bpd = cum_balls / cum_dismissals
        else:
            rpd = cum_runs if cum_runs > 20 else cum_runs*1.5
            bpd = cum_balls if cum_balls > 10 else cum_balls*1.5

        rpb_list.append(rpb)
        bdr_pct_list.append(bdr_pct)
        rpd_list.append(rpd)
        bpd_list.append(bpd)

        # Update cumulative stats with the current match data.
        cum_runs += row['total_runs']
        cum_balls += row['total_balls']
        cum_fours += row['fours']
        cum_sixes += row['sixes']
        cum_dismissals += row['dismissals']

    grp = grp.copy()
    grp['rpb'] = rpb_list
    grp['bdr_pct'] = bdr_pct_list
    grp['rpd'] = rpd_list
    grp['bpd'] = bpd_list
    return grp

result_df = match_stats.groupby(['batter_id', 'bowl_style'], group_keys=False).apply(compute_cumulative_metrics)

# --- Step 4: Add Batter Name ---
# Map the batter_id to batter name using batter_name_map.
result_df['batter'] = result_df['batter_id'].map(batter_name_map)

# Reorder columns as desired:
result_df = result_df[['match_id', 'date', 'batter_id', 'batter', 'bowl_style', 'rpb', 'bdr_pct', 'rpd', 'bpd']]

# Reset index if needed and sort.
result_df.reset_index(drop=True, inplace=True)
result_df.sort_values(by=['date','batter_id', 'bowl_style'], inplace=True)

# The playing XI files has data from IPL 2025 as well, so to drop such rows remove rows where date is null
# This works because the match_stats DataFrame has a date column that is filled with dates from bb_df.
result_df.dropna(subset=['date'],inplace=True)

# --- Step 5: Z-score Standardization and Batter Performance Metrics Calculation---
# List of metrics and their corresponding weights
metrics = {
    'rpb': 0.35,
    'bdr_pct': 0.2,
    'rpd': 0.25,
    'bpd': 0.2
}
# Create a copy of the dataframe or work on it directly if you don't need the original metrics
df_bat = result_df.copy()

# Standardize each metric column using Z-score standardization:
for col in metrics.keys():
    mean_val = df_bat[col].mean()
    std_val = df_bat[col].std()
    # Create a new column for standardized value
    df_bat[col + '_z'] = (df_bat[col] - mean_val) / std_val

# Compute batter quality score using the weighted sum of the standardized metrics:
# batter_quality_score = 0.3 * rpb_z + 0.3 * bdr_pct_z + 0.2 * rpd_z + 0.2 * bpd_z
df_bat['batter_perf_metric'] = (metrics['rpb'] * df_bat['rpb_z'] +
                              metrics['bdr_pct'] * df_bat['bdr_pct_z'] +
                              metrics['rpd'] * df_bat['rpd_z'] +
                              metrics['bpd'] * df_bat['bpd_z'])

# Drop the intermediate standardized columns if you no longer need them:
df_bat.drop(columns=[col+'_z' for col in metrics.keys()], inplace=True)

In [227]:
df_bat[df_bat.batter_id == 5334].tail(7)

,match_id,date,batter_id,batter,bowl_style,rpb,bdr_pct,rpd,bpd,batter_perf_metric
1438,1304093,2022-05-02,5334,Aaron Finch,SLA,1.265306,16.326531,19.076923,15.076923,0.344298
979,1304099,2022-05-07,5334,Aaron Finch,LAF,1.294498,19.741100,40.000000,30.900000,0.860926
1071,1304099,2022-05-07,5334,Aaron Finch,LWS,1.800000,20.000000,18.000000,10.000000,0.610883
1163,1304099,2022-05-07,5334,Aaron Finch,RAF,1.233799,17.153748,23.682927,19.195122,0.447980
1255,1304099,2022-05-07,5334,Aaron Finch,RLB,1.458101,20.670391,23.727273,16.272727,0.592678
1347,1304099,2022-05-07,5334,Aaron Finch,ROB,1.316176,17.647059,35.800000,27.200000,0.738323
1439,1304099,2022-05-07,5334,Aaron Finch,SLA,1.265306,16.326531,19.076923,15.076923,0.344298


### TASK 2 - Computing Bowler Quality Scores

In [228]:
bowl_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\mbm\ipl\combined_ipl_mbm_bowl.csv")

# --- Preparatory Steps ---

# Ensure date is in datetime format if not already.
bowl_df['date'] = pd.to_datetime(bowl_df['date'])

# Sort bowl_df by bowler and date so that historical metrics are computed in order.
bowl_df = bowl_df.sort_values(['bowler_id', 'date'])

# --- Step 1: Aggregate & compute cumulative historical stats for each bowler ---
# First, group the scorecard by match and bowler_id so that we get one row per bowler per match.
match_bowler = bowl_df.groupby(['match_id', 'bowler_id', 'bowler', 'date']).agg({
    'runs': 'sum',
    'wickets': 'sum',
    'overs': 'sum',
    'balls': 'sum'
}).reset_index()

# For cumulative stats we need a match counter per bowler.
match_bowler['match_count'] = 1

# Compute cumulative statistics for each bowler (each row's metrics are computed only using matches BEFORE the current match)
def compute_cumulative_bowling(grp):
    cum_runs = 0
    cum_wickets = 0
    cum_overs = 0
    cum_balls = 0
    cum_matches = 0

    avg_list = []
    sr_list = []
    wp_match_list = []
    op_match_list = []
    cum_match_list = []

    # Iterate row by row for this bowler (already sorted by date)
    for idx, row in grp.iterrows():
        # Use historical cumulative stats from previous matches to compute metrics.
        if cum_wickets > 0:
            # Bowling average: runs conceded per wicket (lower is better)
            avg = cum_runs / cum_wickets
            # Strike rate: balls per wicket (lower is better)
            sr = cum_balls / cum_wickets
        else:
            avg = cum_runs if cum_runs > 30 else cum_runs * 1.4
            sr = cum_balls if cum_balls > 25 else cum_balls * 1.4

        # Wickets per match (higher is better)
        if cum_matches > 0:
            wp_match = cum_wickets / cum_matches
            op_match = cum_overs / cum_matches
        else:
            wp_match = np.nan
            op_match = np.nan

        avg_list.append(avg)
        sr_list.append(sr)
        wp_match_list.append(wp_match)
        op_match_list.append(op_match)
        cum_match_list.append(cum_matches)

        # Update cumulative stats with current match data (so that next row uses these)
        cum_runs += row['runs']
        cum_wickets += row['wickets']
        cum_overs += row['overs']
        cum_balls += row['balls']
        cum_matches += 1

    grp = grp.copy()
    grp['bowling_avg_hist'] = avg_list
    grp['strike_rate_hist'] = sr_list
    grp['wickets_per_match'] = wp_match_list
    grp['overs_per_match'] = op_match_list
    grp['matches_played'] = cum_match_list
    return grp

cum_bowl = match_bowler.groupby('bowler_id', group_keys=False).apply(compute_cumulative_bowling)

# Keep only rows with historical stats already computed.
cum_bowl[['bowling_avg_hist', 'strike_rate_hist', 'wickets_per_match', 'overs_per_match']] = \
    cum_bowl[['bowling_avg_hist', 'strike_rate_hist', 'wickets_per_match', 'overs_per_match']].fillna(0)

# --- Step 2: Normalization of Metrics with Inverse/Direct Transformations ---

# Define a helper function for min–max normalization.
def min_max_norm(series, inverse=False):
    min_val = series.min()
    max_val = series.max()
    # If all values are the same:
    if max_val == min_val:
        return series.apply(lambda x: 0.0 if x == 0 else 1.0)
    if inverse:
        return (max_val - series) / (max_val - min_val)
    else:
        return (series - min_val) / (max_val - min_val)

# Normalized metrics:
cum_bowl['norm_avg'] = min_max_norm(cum_bowl['bowling_avg_hist'], inverse=True)   # lower is better
cum_bowl['norm_sr']  = min_max_norm(cum_bowl['strike_rate_hist'], inverse=True)    # lower is better
cum_bowl['norm_wp']  = min_max_norm(cum_bowl['wickets_per_match'], inverse=False)  # higher is better
cum_bowl['norm_op']  = min_max_norm(cum_bowl['overs_per_match'], inverse=False)    # higher is better

# --- Step 3: Compute the Bowling Quality Score Using Weights ---
# Weights: Bowling Average: 0.2, Strike Rate: 0.2, Wickets per Match: 0.3, Overs per Match: 0.3
cum_bowl['bowler_quality_score'] = (0.2 * cum_bowl['norm_avg'] +
                                    0.2 * cum_bowl['norm_sr'] +
                                    0.3 * cum_bowl['norm_wp'] +
                                    0.3 * cum_bowl['norm_op'])

# Apply credibility adjustment:
# Suppose full credibility is given after 45 matches.
credibility_weight = np.minimum(1, cum_bowl['matches_played'] / 45)
cum_bowl['bowler_quality_score'] = cum_bowl['bowler_quality_score'] * credibility_weight

# --- Finalize the DataFrame ---
final_bowler_df = cum_bowl[['match_id', 'date', 'bowler_id', 'bowler',
                             'bowling_avg_hist', 'strike_rate_hist', 'wickets_per_match', 'overs_per_match',
                             'bowler_quality_score']].copy()

# Penalize part-time or non-bowlers.
def adjust_for_part_time(row):
    if row['overs_per_match'] < 0.4:
        return row['bowler_quality_score'] * (row['overs_per_match'] / 0.4)
    else:
        return row['bowler_quality_score']

final_bowler_df.loc[:, 'bowler_quality_score'] = final_bowler_df.apply(adjust_for_part_time, axis=1)

# --- Final Normalization Step ---
# Re-normalize bowler_quality_score so that its min is 0 and max is 1.
min_score = final_bowler_df['bowler_quality_score'].min()
max_score = final_bowler_df['bowler_quality_score'].max()
final_bowler_df.loc[:, 'bowler_quality_score'] = (final_bowler_df['bowler_quality_score'] - min_score) / (max_score - min_score)

### TASK 3 - Compute Bowler Type Proportion in Opposition

In [229]:
final_bowler_df

,match_id,date,bowler_id,bowler,bowling_avg_hist,strike_rate_hist,wickets_per_match,overs_per_match,bowler_quality_score
0,335982,2008-04-18,5766,David Hussey,0.000000,0.000000,0.000000,0.000000,0.000000
1,335982,2008-04-18,6938,Ashley Noffke,0.000000,0.000000,0.000000,0.000000,0.000000
2,335982,2008-04-18,7133,Ricky Ponting,0.000000,0.000000,0.000000,0.000000,0.000000
3,335982,2008-04-18,8291,Cameron White,0.000000,0.000000,0.000000,0.000000,0.000000
4,335982,2008-04-18,26184,Ajit Agarkar,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
24333,1426312,2024-05-26,1159711,Shahbaz Ahmed,40.750000,26.700000,0.370370,1.611111,0.650376
24334,1426312,2024-05-26,1175485,Abdul Samad,56.500000,27.500000,0.040816,0.183673,0.217138
24335,1426312,2024-05-26,1175496,Nitish Kumar Reddy,69.000000,39.333333,0.214286,1.292857,0.172667
24336,1426312,2024-05-26,1209292,Vaibhav Arora,31.000000,22.000000,0.947368,3.184211,0.362549


In [230]:
# Step 1: Setup mapping from bowler_id to bowl_style
bowler_style_map = bb_df.dropna(subset=['bowl_style']) \
                         .groupby('bowler_id')['bowl_style'] \
                         .first().to_dict()

final_bowler_df['bowl_style'] = final_bowler_df['bowler_id'].map(bowler_style_map)
final_bowler_df['bowl_style'] = final_bowler_df['bowl_style'].fillna('Unknown')

# Step 2: Get opposition team
def get_opposition_team(match_id, batter_id):
    row = xi_df[xi_df['match_id'] == match_id].iloc[0]
    team1 = [int(x) for x in row['team1_playing_xi']]
    team2 = [int(x) for x in row['team2_playing_xi']]
    if batter_id in team1:
        return team2
    elif batter_id in team2:
        return team1
    else:
        return []

# Step 3: Compute normalized bowler type proportions per batter per match
def compute_bowler_proportions_for_batter(match_id, batter_id):
    opposition_ids = get_opposition_team(match_id, batter_id)

    if not opposition_ids:
        return {}

    opp_bowlers = final_bowler_df[(final_bowler_df['match_id'] == match_id) &
                                  (final_bowler_df['bowler_id'].isin(opposition_ids))]

    if opp_bowlers.empty:
        return {}

    # Group by bowl_style and sum quality scores
    style_quality = opp_bowlers.groupby('bowl_style')['bowler_quality_score'].sum()

    total_quality = style_quality.sum()
    if total_quality == 0:
        return {}

    # Normalize so proportions sum to 1
    normalized_proportions = (style_quality / total_quality).to_dict()
    return normalized_proportions

# Step 4: Apply to df_bat
# For performance, we'll first compute a lookup dictionary
from collections import defaultdict

# (match_id, batter_id) → dict of bowl_style proportions
batter_match_to_proportion = {}

for (match_id, batter_id) in df_bat[['match_id', 'batter_id']].drop_duplicates().itertuples(index=False):
    batter_match_to_proportion[(match_id, batter_id)] = compute_bowler_proportions_for_batter(match_id, batter_id)

# Step 5: Use this to fill in the proportion column for the row's target bowl_style
def get_proportion(row):
    props = batter_match_to_proportion.get((row['match_id'], row['batter_id']), {})
    return props.get(row['bowl_style'], 0.0)

df_bat['bowler_proportion'] = df_bat.apply(get_proportion, axis=1)

In [231]:
df_bat.sample()

,match_id,date,batter_id,batter,bowl_style,rpb,bdr_pct,rpd,bpd,batter_perf_metric,bowler_proportion
69161,1216509,2020-10-17,234675,Ravindra Jadeja,LAF,1.411255,16.883117,23.285714,16.5,0.49542,0.0


In [232]:
df_bat[df_bat.match_id == 1216510]

,match_id,date,batter_id,batter,bowl_style,rpb,bdr_pct,rpd,bpd,batter_perf_metric,bowler_proportion
964,1216510,2020-09-24,5334,Aaron Finch,LAF,1.342412,21.400778,57.500000,42.833333,1.266345,0.021451
1056,1216510,2020-09-24,5334,Aaron Finch,LWS,1.800000,20.000000,18.000000,10.000000,0.610883,0.000000
1148,1216510,2020-09-24,5334,Aaron Finch,RAF,1.252294,17.278287,22.135135,17.675676,0.423043,0.448927
1240,1216510,2020-09-24,5334,Aaron Finch,RLB,1.429752,18.181818,24.714286,17.285714,0.555451,0.249360
1332,1216510,2020-09-24,5334,Aaron Finch,ROB,1.267717,17.322835,32.200000,25.400000,0.643686,0.280261
...,...,...,...,...,...,...,...,...,...,...,...
146060,1216510,2020-09-24,1175441,Ravi Bishnoi,LWS,0.000000,0.000000,0.000000,0.000000,-1.016245,0.000000
146130,1216510,2020-09-24,1175441,Ravi Bishnoi,RAF,0.250000,0.000000,1.500000,6.000000,-0.796258,0.614218
146200,1216510,2020-09-24,1175441,Ravi Bishnoi,RLB,0.000000,0.000000,0.000000,0.000000,-1.016245,0.241449
146270,1216510,2020-09-24,1175441,Ravi Bishnoi,ROB,0.000000,0.000000,0.000000,0.000000,-1.016245,0.118781


In [233]:
df_bat.batter_perf_metric.max(), df_bat.batter_perf_metric.min(), df_bat.batter_perf_metric.mean(), df_bat.batter_perf_metric.std()

(5.321343123628609,
 -1.0162449303026222,
 1.0264334673433548e-16,
 0.8673093794300385)

### TASK 4 - Compute Batter Matchup Score

In [234]:
# --- Task 4: Compute Batter Matchup Score ---

# Step 1: For each batter (per match) compute the weighted performance metric:
#         weighted_perf = sum over bowler types (bowler_proportion * batter_perf_metric)
weighted_perf_df = (
    df_bat.groupby(['match_id', 'batter_id', 'batter'], as_index=False)
          .apply(lambda g: pd.Series({'weighted_perf': (g['bowler_proportion'] * g['batter_perf_metric']).sum()}))
)
weighted_perf_df.reset_index(drop=True, inplace=True)

# Step 2: Linearly map the weighted_perf values to a 1–10 scale.
#         That is, apply a min–max normalization where min value maps to 1 and max to 10.
min_val = weighted_perf_df['weighted_perf'].min()
max_val = weighted_perf_df['weighted_perf'].max()

# In case max_val equals min_val (degenerate case), we can simply assign 1 (or any constant) to all.
if max_val == min_val:
    weighted_perf_df['batter_matchup_score'] = 1.0
else:
    weighted_perf_df['batter_matchup_score'] = 1 + 9 * (weighted_perf_df['weighted_perf'] - min_val) / (max_val - min_val)

weighted_perf_df.drop(columns=['weighted_perf'], inplace=True)

### Exporting it to a CSV File

In [235]:
weighted_perf_df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bat_matchup.csv", index=False)